# Session summaries for LongMemEval_s (pilot, 60 questions)

Work kernel for memory-recall-bench: one summary per session with `Qwen/Qwen3-4B-Instruct-2507` on 2x T4. Output is scored offline.

In [ ]:
import urllib.request, os
if not os.path.exists('/kaggle/working/lme.json'):
    urllib.request.urlretrieve('https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json', '/kaggle/working/lme.json')
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers; print(transformers.__version__)

In [ ]:
%%writefile /kaggle/working/worker.py
import json, os, sys, time, urllib.request
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

WORKER, WORKERS = int(sys.argv[1]), int(sys.argv[2])
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
IDS = set(["60036106", "95228167", "8a2466db", "06878be2", "75832dbd", "0edc2aef", "35a27287", "32260d93", "195a1a1b", "afdc33df", "caf03d32", "54026fce", "06f04340", "6b7dfb22", "1a1907b4", "09d032c9", "38146c39", "d24813b1", "57f827a0", "505af2f5", "75f70248", "d6233ab6", "1da05512", "fca70973", "b6025781", "a89d7624", "b0479f84", "1d4e3b97", "07b6f563", "1c0ddc50", "0a34ad58", "edced276", "0ea62687", "efc3f7c2", "a3332713", "099778bb", "36b9f61e", "bf659f65", "gpt4_d12ceb0e", "00ca467f", "a96c20ee", "c4a1ceb8", "61f8c8f8", "b5ef892d", "gpt4_2f8be40d", "d682f1a2", "27016adc", "3fe836c9", "gpt4_5501fe77", "09ba9854_abs", "gpt4_ab202e7f", "e831120c", "gpt4_7fce9456", "gpt4_59c863d7", "4adc0475", "8e91e7d9", "gpt4_e05b82a6", "1a8a66a6", "09ba9854", "3fdac837"])
MAX_SESSION_CHARS = 60_000
TOKEN_BUDGET = 24_000          # prompt tokens per batch
MAX_NEW = 220
PROMPT = (
    "Below is one chat session between a user and an assistant. Write a faithful summary "
    "in at most 120 words. Keep every concrete fact a later question could ask about: names, "
    "numbers, dates, places, preferences, decisions, plans, and what the user said about "
    "themselves. Do not add anything that is not in the session.\n\n<session>\n{body}\n</session>"
)

data = json.loads(Path("/kaggle/working/lme.json").read_text())
sessions = {}
for q in data:
    if q["question_id"] not in IDS:
        continue
    for sid, msgs in zip(q["haystack_session_ids"], q["haystack_sessions"]):
        sessions.setdefault(sid, msgs)
sids = sorted(sessions)[WORKER::WORKERS]
out_path = Path(f"/kaggle/working/summaries_qwen_{WORKER}.jsonl")
done = set()
if out_path.exists():
    for line in out_path.read_text().splitlines():
        try: done.add(json.loads(line)["session_id"])
        except Exception: pass
print(f"worker {WORKER}: {len(sids)} sessions, {len(done)} already done", flush=True)

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda:0").eval()

items = []
for sid in sids:
    if sid in done:
        continue
    body = "\n".join(f"{m.get('role', '?')}: {m.get('content', '')}" for m in sessions[sid])
    cut = len(body) > MAX_SESSION_CHARS
    body = body[:MAX_SESSION_CHARS]
    text = tok.apply_chat_template([{"role": "user", "content": PROMPT.format(body=body)}],
                                   tokenize=False, add_generation_prompt=True)
    items.append((sid, text, cut, len(tok(text).input_ids)))
items.sort(key=lambda x: x[3])

t0, n = time.time(), 0
with out_path.open("a", encoding="utf-8") as out:
    i = 0
    while i < len(items):
        j = i + 1
        while j < len(items) and (j - i + 1) * items[j][3] <= TOKEN_BUDGET and j - i < 16:
            j += 1
        batch = items[i:j]
        enc = tok([b[1] for b in batch], return_tensors="pt", padding=True).to("cuda:0")
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False)
        for b, g in zip(batch, gen[:, enc.input_ids.shape[1]:]):
            out.write(json.dumps({"session_id": b[0], "summary": tok.decode(g, skip_special_tokens=True).strip(),
                                  "truncated": b[2], "input_tokens": b[3], "model": MODEL}) + "\n")
        out.flush()
        n += len(batch); i = j
        print(f"worker {WORKER}: {n}/{len(items)} ({time.time() - t0:.0f}s)", flush=True)
print(f"worker {WORKER}: finished", flush=True)

In [ ]:
import subprocess, os, torch
W = max(1, torch.cuda.device_count())
procs = [subprocess.Popen(['python', '/kaggle/working/worker.py', str(i), str(W)],
          env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(i)}) for i in range(W)]
print([p.wait() for p in procs])
os.remove('/kaggle/working/lme.json')
os.remove('/kaggle/working/worker.py')